# The Office - dobór odcinków

Dzieli ręczną listę odcinków na część deweloperską i testową. Do *The Office* nie ma zbioru adnotacji w rodzaju TVR, więc nie ma czego rankingować: odcinki są wybrane ręcznie, po jednym lub kilku z każdego sezonu, z naciskiem na wyraźną i różnorodną warstwę wizualną (przyjęcia, zawody biurowe, wyjazdy, sceny plenerowe). Lista jest wejściem, nie wynikiem.

Notatnik robi to samo co `tbbt_01_episode_selection` i wypisuje te same tabele w tej samej kolejności, pomijając dwie sekcje, których dla tego serialu nie ma: ranking zapytań `v` i eksport opisów `v`. Podział na `dev` i `test` powstaje tą samą metodą, żeby oba zbiory serialowe były porównywalne.

**Wymaga:** `data/interim/office/office_selection_manual.csv` - ręcznej listy odcinków z kolumnami `episode;season;episode_no;title`.

**Zapisuje:** `data/interim/office/office_selection_<COUNT>.csv` - wybór odcinków z kolumną `split`. `COUNT` w nazwie to liczba wierszy w pliku ręcznym, więc nie trzeba jej nigdzie wpisywać.

**Dalej:** adnotowanie odcinków w `tools/interval-annotator`, a potem `office_02_annotations.ipynb`, ostatni notatnik tej ścieżki (TBBT ma tu jeden krok więcej).

## 1. Dobór odcinków i podział na dev/test

**Zapisuje:** `data/interim/office/office_selection_<COUNT>.csv` - wybrane odcinki z kolumną `split`.

Lista uporządkowana chronologicznie dzielona jest na `DEV_COUNT` równych bloków, a z każdego brany jest środek. Metoda jest deterministyczna, bez ziarna losowania, więc każde uruchomienie daje ten sam podział, a `dev` rozkłada się po całej rozpiętości serialu.

Zmiana podziału unieważniłaby adnotacje już zrobione, więc blok porównuje wynik z zapisanym plikiem i wypisuje różnice, zamiast je po cichu nadpisać.

In [ ]:
DEV_COUNT = 6            # how many of the selected episodes go to the dev split

import csv
from pathlib import Path

ROOT     = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
DATA_DIR = ROOT / "data" / "interim" / "office"
MANUAL_CSV = DATA_DIR / "office_selection_manual.csv"

with open(MANUAL_CSV, encoding="utf-8-sig", newline="") as f:
    manual = list(csv.DictReader(f, delimiter=";"))
title = {r["episode"]: r.get("title", "") for r in manual}
season = {r["episode"]: r["season"] for r in manual}
number = {r["episode"]: int(r["episode_no"]) for r in manual}

EPISODE_COUNT = len(manual)
SELECTION_CSV = DATA_DIR / f"office_selection_{EPISODE_COUNT}.csv"

# dev: middle of each of DEV_COUNT equal blocks of the chronological list
chronological = sorted(title)
step = len(chronological) / DEV_COUNT
dev = {chronological[int((i + 0.5) * step)] for i in range(DEV_COUNT)}
split = {ep: ("dev" if ep in dev else "test") for ep in chronological}

# a changed split would invalidate annotations already made
if SELECTION_CSV.exists():
    with open(SELECTION_CSV, encoding="utf-8-sig", newline="") as f:
        previous = {r["episode"]: r["split"] for r in csv.DictReader(f, delimiter=";")}
    changed = {ep: (previous.get(ep, "-"), split[ep])
               for ep in split if previous.get(ep) != split[ep]}
    dropped = sorted(set(previous) - set(split))
    if changed or dropped:
        print(f"WARNING: the split differs from {SELECTION_CSV.name}")
        for ep, (was, now) in sorted(changed.items()):
            print(f"   {ep}: {was} -> {now}")
        for ep in dropped:
            print(f"   {ep}: gone from the manual selection")
        print("   annotations already made were prepared under the previous split\n")
    else:
        print(f"split identical to the saved {SELECTION_CSV.name}\n")

with open(SELECTION_CSV, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["episode", "season", "episode_no", "split"])
    for ep in chronological:
        w.writerow([ep, season[ep], number[ep], split[ep]])


def summarize(label, items):
    seasons = sorted({season[e] for e in items})
    print(f"{label:<6}{len(items):>4} episodes   "
          f"seasons: {', '.join(s[1:] for s in seasons)}")


print(f"selected {len(chronological)} episodes from the manual list")
summarize("total", chronological)
summarize("dev", [e for e in chronological if e in dev])
summarize("test", [e for e in chronological if e not in dev])

print(f"\n{'episode':<9}{'split':<7}{'season':>8}{'no':>5}")
for ep in chronological:
    print(f"{ep:<9}{split[ep]:<7}{season[ep]:>8}{number[ep]:>5}")
print(f"\nsaved -> {SELECTION_CSV.relative_to(ROOT)}")

## 2. Tytuły odcinków

Tytuły brane są z ręcznej listy `office_selection_manual.csv`; dla TBBT ta sama tabela powstaje z bufora albo z TVmaze.

In [ ]:
print(f"{'episode':<10}{'split':<7}title")
print("-" * 52)
for ep in chronological:
    print(f"{ep:<10}{split[ep]:<7}{title[ep]}")